# J2S4 — LLM & Hugging Face : Explicabilité en langage naturel
## BankRisk Intelligence Platform · Contexte bancaire ivoirien

**Objectif :** Intégrer un LLM (Mistral 7B via Hugging Face) pour produire des explications  
de décision de crédit en français naturel, complémentaires au waterfall SHAP.  
**Entrée :** `credit_risk_clean.parquet` + modèles `.joblib` (J2S3).  
**Livrable :** `utils/llm_utils.py` — module réutilisable par J3S2 (Streamlit).

---

### Principe : SHAP + LLM = double lisibilité

```
Waterfall SHAP      →  lisibilité graphique  (analyste technique)
Explication LLM     →  lisibilité texte      (comité de crédit, client)
```

> Le waterfall SHAP montre **quelles features** ont contribué et **dans quel sens**.  
> L'explication LLM traduit ce graphique en **phrases compréhensibles** par un  
> décideur non technicien ou un client qui conteste une décision.

### Architecture Hugging Face LLM

> L'**API Hugging Face Inference** (router `router.huggingface.co/v1`) est compatible  
> OpenAI SDK et permet d'appeler des modèles hébergés (ici Mistral 7B Instruct v0.2).  
> Authentification via token HF stocké dans `.env` — jamais en dur dans le code.  
> Fallback propre si token absent : waterfall SHAP seul, sans erreur bloquante.  
> En J3S2, le module `utils/llm_utils.py` sera importé par l'app Streamlit.

### Conformité BCEAO

> L'explication LLM est **complémentaire**, pas substitutive — le waterfall SHAP  
> reste le livrable de conformité principal (Instruction n°026-2016). Le texte LLM  
> facilite la communication de la décision auprès du client ou du comité.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 0 — Setup · ROOT detection + pip install + imports (cellule unique)
# ═══════════════════════════════════════════════════════════════════════════
import sys, os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/bankrisk')
except ImportError:
    IN_COLAB = False
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'data').exists() or (ROOT / 'requirements.txt').exists():
            break
        ROOT = ROOT.parent

print(f"Environnement : {'Google Colab' if IN_COLAB else 'VS Code local'}")
print(f"ROOT : {ROOT}")

# ── Chargement .env (token Hugging Face) ──────────────────────────────────────
try:
    from dotenv import load_dotenv
    _env_path = ROOT / '.env'
    if _env_path.exists():
        load_dotenv(_env_path)
        print(f"✓ .env chargé depuis : {_env_path}")
    else:
        print(f"⚠ .env absent de {ROOT} — HF_TOKEN non chargé")
        print("  Créer le fichier .env avec : HF_TOKEN=votre_token_hf")
except ImportError:
    print("⚠ python-dotenv absent — pip install python-dotenv")

# ── Installation des dépendances ──────────────────────────────────────────────
req_file = ROOT / 'requirements.txt'
if req_file.exists():
    os.system(f'{sys.executable} -m pip install -r {req_file} -q')
else:
    os.system(
        f'{sys.executable} -m pip install '
        'pandas numpy scikit-learn shap joblib plotly pyarrow '
        'openai python-dotenv imbalanced-learn -q'
    )

# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib
import shap
import json as json_lib
import py_compile
import time
import importlib.util
from openai import OpenAI

# ── Constantes globales ───────────────────────────────────────────────────────
MODELS_DIR = ROOT / 'models'
UTILS_DIR  = ROOT / 'utils'

HF_MODEL    = "mistralai/Mistral-7B-Instruct-v0.2:featherless-ai"
HF_BASE_URL = "https://router.huggingface.co/v1"

RANDOM_STATE     = 42
SEUIL_PRODUCTION = 0.42

FEATURES_OFFICIELLES = [
    'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate',
    'loan_percent_income', 'cb_person_cred_hist_length',
    'home_RENT', 'home_MORTGAGE', 'home_OWN', 'default_enc',
    'debt_service_rate', 'monthly_payment_proxy', 'log_income', 'high_risk_intent',
]

# ── Chargement du token ───────────────────────────────────────────────────────
HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    print(f"✓ HF_TOKEN chargé ({len(HF_TOKEN)} caractères)")
else:
    print("⚠ HF_TOKEN non trouvé — les tests API seront ignorés")
    print("  VS Code  : ajouter HF_TOKEN=<token> dans .env puis relancer")
    print("  Colab    : Secrets → ajouter HF_TOKEN")

print("\n✓ Setup complet — J2S4 prêt")


---
## Bloc 1 — Test de l'API Hugging Face (hors Streamlit)

> Avant de packager la fonction dans un module, on valide l'appel API depuis le notebook.  
> Le token HF se crée sur `https://huggingface.co/settings/tokens` (Fine-grained ou Write).  
> Il doit être stocké dans `.env` (VS Code) ou dans les Secrets Colab — jamais en dur.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 1 — Test direct de l'API Hugging Face Mistral 7B
# ═══════════════════════════════════════════════════════════════════════════

# ─── Récupération token (Colab : Secrets, VS Code : .env via Cell 0) ──────────
if IN_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        print("✓ Token chargé depuis Colab Secrets")
    except Exception:
        print("⚠ HF_TOKEN non trouvé dans Colab Secrets")
        print("  Colab → icône clé → Secrets → ajouter HF_TOKEN")

# ─── Client API (compatible OpenAI SDK) ───────────────────────────────────────
def get_hf_client(token: str) -> OpenAI:
    """Crée un client OpenAI pointant vers le router Hugging Face."""
    return OpenAI(base_url=HF_BASE_URL, api_key=token)

# ─── Test complet ─────────────────────────────────────────────────────────────
if HF_TOKEN:
    client = get_hf_client(HF_TOKEN)
    try:
        response = client.chat.completions.create(
            model=HF_MODEL,
            messages=[
                {"role": "system",  "content": "Tu es un assistant bancaire ivoirien."},
                {"role": "user",    "content": "En une phrase, qu'est-ce que le taux d'endettement ?"},
            ],
            max_tokens=150,
            temperature=0.3,
        )
        print("✓ API Hugging Face opérationnelle")
        print(f"  Modèle  : {HF_MODEL}")
        print(f"  Réponse : {response.choices[0].message.content.strip()}")
    except Exception as e:
        print(f"✗ Erreur API : {e}")
        print("  → Exécuter Bloc 1b pour diagnostiquer")
else:
    print("⚠ Test ignoré — HF_TOKEN absent")
    print("  L'app Streamlit (J3S2) fonctionnera en mode dégradé (SHAP seul)")


---
## Bloc 1b — Diagnostic connectivité Hugging Face

> Exécuter **uniquement si Bloc 1 retourne une erreur**.  
> Effectue un appel minimal (`max_tokens=5`) pour isoler la cause de l'échec.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 1b — Diagnostic : test minimal sur le router HF (max_tokens=5)
# ═══════════════════════════════════════════════════════════════════════════
print("Diagnostic de connexion au router Hugging Face...")
print(f"  Token chargé : {'oui ✓' if HF_TOKEN else 'NON ✗'}")
print(f"  Base URL     : {HF_BASE_URL}")
print(f"  Modèle       : {HF_MODEL}")
print()

if not HF_TOKEN:
    print("✗ HF_TOKEN absent — vérifier le fichier .env et relancer Cell 0")
else:
    try:
        client_diag = get_hf_client(HF_TOKEN)
        resp = client_diag.chat.completions.create(
            model    = HF_MODEL,
            messages = [{"role": "user", "content": "Réponds juste: ok"}],
            max_tokens  = 5,
            temperature = 0.1,
        )
        print("✓ Connexion router HF opérationnelle")
        print(f"  Réponse minimale : {resp.choices[0].message.content.strip()}")
        print()
        print("→ Le token est valide. Relancer Bloc 1 pour le test complet.")

    except Exception as e:
        err = str(e)
        print(f"✗ Erreur : {err}")
        print()
        if "401" in err or "unauthorized" in err.lower():
            print("  Cause : token HF expiré ou invalide")
            print("  → Régénérer sur https://huggingface.co/settings/tokens")
            print("  → Mettre à jour .env puis relancer Cell 0")
        elif "404" in err:
            print("  Cause : modèle non disponible via ce provider")
            print("  → Vérifier que HF_MODEL contient bien le suffixe :featherless-ai")
        elif "connection" in err.lower():
            print("  Cause : pas d'accès réseau à router.huggingface.co")
            print("  → Vérifier la connexion internet ou le proxy")
        else:
            print("  → Vérifier HF_BASE_URL et HF_MODEL dans Cell 0")


---
## Bloc 2 — Fonctions d'explication LLM

> Deux fonctions complémentaires :
> - `build_shap_prompt()` — construit le prompt à partir des contributions SHAP
> - `generate_shap_explanation()` — appelle l'API HF et retourne l'explication texte
>
> Ces fonctions seront exportées dans `utils/llm_utils.py` (Bloc 3)  
> puis importées par l'app Streamlit en **J3S2**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 2 — Fonctions build_shap_prompt() et generate_shap_explanation()
# ═══════════════════════════════════════════════════════════════════════════

def build_shap_prompt(
    shap_contributions: list,
    decision: str,
    rf_proba: float,
    seuil: float,
) -> str:
    # Construit le prompt LLM a partir des contributions SHAP.
    # shap_contributions : liste de dict {feature, shap_value, raw_value}
    #                      triee par |shap_value| decroissant
    # decision           : 'REFUS' ou 'APPROBATION'
    # rf_proba           : probabilite de defaut [0, 1]
    # seuil              : seuil de production (ex. 0.42)
    # Retourne           : prompt complet pret a envoyer au LLM

    contrib_lines = []
    for c in shap_contributions[:6]:
        direction = "augmente" if c['shap_value'] > 0 else "reduit"
        impact    = "fortement" if abs(c['shap_value']) > 0.05 else "moderement"
        contrib_lines.append(
            f"- {c['feature']} = {c['raw_value']:.3g} "
            f"({direction} {impact} le risque, contribution = {c['shap_value']:+.3f})"
        )

    contrib_text  = "\n".join(contrib_lines)
    decision_noun = "refus" if decision == "REFUS" else "approbation"

    prompt = (
        "Tu es un analyste credit dans une banque ivoirienne (contexte UEMOA/BCEAO).\n"
        "Un modele de scoring a produit la decision suivante pour un dossier de credit :\n\n"
        f"DECISION : {decision} "
        f"(probabilite de defaut = {rf_proba:.1%}, seuil = {seuil:.0%})\n\n"
        f"FACTEURS CLES (contributions SHAP) :\n{contrib_text}\n\n"
        f"Redige une explication claire et professionnelle de cette decision de {decision_noun}, "
        "en 3 a 4 phrases maximum. L'explication doit :\n"
        "1. Mentionner les 2-3 facteurs les plus determinants\n"
        "2. Etre comprehensible par un client ou un membre du comite de credit\n"
        "3. Ne pas utiliser de jargon technique (pas de 'SHAP', pas de 'modele ML')\n"
        "4. Etre redigee en francais, adapte au contexte bancaire ivoirien\n\n"
        "Reponds uniquement avec l'explication, sans introduction ni conclusion."
    )
    return prompt


def generate_shap_explanation(
    shap_contributions: list,
    decision: str,
    rf_proba: float,
    seuil: float,
    hf_token: str,
    model: str = HF_MODEL,
    base_url: str = HF_BASE_URL,
    max_retries: int = 3,
    retry_delay: float = 8.0,
) -> tuple:
    # Appelle l'API Hugging Face Mistral 7B pour generer l'explication texte.
    # Retourne (str, None) si succes, (None, str) si echec (mode degrade).
    #
    # Le router HF Inference Providers ne supporte pas le parametre legacy
    # `wait_for_model` de l'ancienne API d'inference : la seule parade cote
    # client contre les erreurs "model is busy" (frequentes sur les
    # providers gratuits type featherless-ai, souvent un cold-start
    # transitoire) est de reessayer avec un delai.

    if not hf_token:
        return None, "Token Hugging Face absent — mode degrade (SHAP seul)"

    client   = OpenAI(base_url=base_url, api_key=hf_token)
    prompt   = build_shap_prompt(shap_contributions, decision, rf_proba, seuil)
    messages = [
        {
            "role":    "system",
            "content": (
                "Tu es un analyste credit senior dans une banque ivoirienne. "
                "Tu rediges des explications de decisions de credit claires "
                "et professionnelles, en francais."
            ),
        },
        {"role": "user", "content": prompt},
    ]

    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model, messages=messages, max_tokens=300, temperature=0.3,
            )
            return response.choices[0].message.content.strip(), None
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    return None, f"Erreur API Hugging Face : {str(last_error)}"


# ─── Test des fonctions ────────────────────────────────────────────────────────
test_contributions = [
    {'feature': 'monthly_payment_proxy', 'shap_value':  0.18, 'raw_value': 2.4},
    {'feature': 'loan_int_rate',         'shap_value':  0.12, 'raw_value': 14.5},
    {'feature': 'person_income',         'shap_value': -0.08, 'raw_value': 35000},
    {'feature': 'debt_service_rate',     'shap_value':  0.06, 'raw_value': 0.29},
    {'feature': 'home_RENT',             'shap_value':  0.05, 'raw_value': 1},
]

# Affichage du prompt construit (sans appel API)
prompt_exemple = build_shap_prompt(test_contributions, "REFUS", 0.71, 0.42)
print("=" * 65)
print("Prompt genere par build_shap_prompt() :")
print("=" * 65)
print(prompt_exemple)
print("=" * 65)

# Appel API si token disponible
if HF_TOKEN:
    explication, erreur = generate_shap_explanation(
        shap_contributions = test_contributions,
        decision           = "REFUS",
        rf_proba           = 0.71,
        seuil              = SEUIL_PRODUCTION,
        hf_token           = HF_TOKEN,
    )
    if explication:
        print("\n✓ Explication LLM generee :")
        print("-" * 65)
        print(explication)
        print("-" * 65)
    else:
        print(f"\n⚠ Mode degrade : {erreur}")
else:
    print("\n⚠ Test API ignore — HF_TOKEN absent")
    print("  Exemple de sortie attendue :")
    print("  'Le dossier a ete refuse principalement en raison d'une charge")
    print("   mensuelle elevee par rapport aux revenus declares, combinee a")
    print("   un taux d'interet important. Ces indicateurs traduisent un")
    print("   risque de surendettement au regard des normes UEMOA.'")


---
## Bloc 3 — Export du module `utils/llm_utils.py`

> Les fonctions validées en Bloc 2 sont packagées dans un module autonome.  
> Ce module sera importé directement par l'app Streamlit en **J3S2** :
>
> ```python
> from utils.llm_utils import generate_shap_explanation, HF_MODEL
> ```
>
> Avantage : le code LLM n'est écrit **qu'une seule fois** — testé ici, réutilisé en J3S2.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOC 3 — Génération de utils/llm_utils.py (module réutilisable J3S2)
# ═══════════════════════════════════════════════════════════════════════════

UTILS_DIR.mkdir(parents=True, exist_ok=True)

utils_path = UTILS_DIR / 'llm_utils.py'

MODULE_SOURCE = '''"""
utils/llm_utils.py — BankRisk Intelligence Platform
Fonctions LLM pour l'explicabilite en langage naturel.
Utilise par : J3S2 (Streamlit), tests unitaires.

Dependances : openai>=1.0
Token        : HF_TOKEN dans .env (VS Code) ou st.secrets (Streamlit)
"""
import time
from openai import OpenAI

# Constantes API
HF_MODEL    = "mistralai/Mistral-7B-Instruct-v0.2:featherless-ai"
HF_BASE_URL = "https://router.huggingface.co/v1"


def get_hf_client(token: str) -> OpenAI:
    """Cree un client OpenAI pointant vers le router Hugging Face."""
    return OpenAI(base_url=HF_BASE_URL, api_key=token)


def build_shap_prompt(
    shap_contributions: list,
    decision: str,
    rf_proba: float,
    seuil: float,
) -> str:
    """Construit le prompt LLM a partir des contributions SHAP."""
    contrib_lines = []
    for c in shap_contributions[:6]:
        direction = "augmente" if c["shap_value"] > 0 else "reduit"
        impact    = "fortement" if abs(c["shap_value"]) > 0.05 else "moderement"
        contrib_lines.append(
            f"- {c['feature']} = {c['raw_value']:.3g} "
            f"({direction} {impact} le risque, contribution = {c['shap_value']:+.3f})"
        )

    contrib_text  = "\\n".join(contrib_lines)
    decision_noun = "refus" if decision == "REFUS" else "approbation"

    return (
        "Tu es un analyste credit dans une banque ivoirienne (contexte UEMOA/BCEAO).\\n"
        "Un modele de scoring a produit la decision suivante :\\n\\n"
        f"DECISION : {decision} "
        f"(probabilite de defaut = {rf_proba:.1%}, seuil = {seuil:.0%})\\n\\n"
        f"FACTEURS CLES (contributions SHAP) :\\n{contrib_text}\\n\\n"
        f"Redige une explication de {decision_noun} en 3-4 phrases max. "
        "Mentionner les 2-3 facteurs cles. "
        "Pas de jargon technique. En francais, contexte bancaire ivoirien.\\n\\n"
        "Reponds uniquement avec l'explication."
    )


def generate_shap_explanation(
    shap_contributions: list,
    decision: str,
    rf_proba: float,
    seuil: float,
    hf_token: str,
    model: str = HF_MODEL,
    base_url: str = HF_BASE_URL,
    max_retries: int = 3,
    retry_delay: float = 8.0,
) -> tuple:
    """Appelle HF Mistral 7B — retourne (explication, None) ou (None, erreur).

    Le router HF Inference Providers (router.huggingface.co) ne supporte pas
    le parametre legacy `wait_for_model` de l'ancienne API d'inference : la
    seule parade cote client contre les erreurs "model is busy" (frequentes
    sur les providers gratuits type featherless-ai, souvent un cold-start
    transitoire) est de reessayer avec un delai.
    """
    if not hf_token:
        return None, "Token HF absent — mode degrade (SHAP seul)"

    client   = OpenAI(base_url=base_url, api_key=hf_token)
    prompt   = build_shap_prompt(shap_contributions, decision, rf_proba, seuil)
    messages = [
        {
            "role":    "system",
            "content": (
                "Tu es un analyste credit senior dans une banque ivoirienne. "
                "Tu rediges des explications de decisions de credit claires "
                "et professionnelles, en francais."
            ),
        },
        {"role": "user", "content": prompt},
    ]

    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model, messages=messages, max_tokens=300, temperature=0.3,
            )
            return response.choices[0].message.content.strip(), None
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    return None, f"Erreur API Hugging Face : {str(last_error)}"
'''

with open(utils_path, 'w', encoding='utf-8') as f:
    f.write(MODULE_SOURCE)

print(f"✓ Module généré : {utils_path}")
print(f"  Lignes : {sum(1 for _ in open(utils_path))}")

# Validation syntaxique
try:
    py_compile.compile(str(utils_path), doraise=True)
    print("✓ Syntaxe Python valide")
except py_compile.PyCompileError as e:
    print(f"✗ Erreur de syntaxe : {e}")
    raise

# Test d'import
spec   = importlib.util.spec_from_file_location("llm_utils", utils_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

assert hasattr(module, 'get_hf_client'),            "✗ get_hf_client manquant"
assert hasattr(module, 'build_shap_prompt'),         "✗ build_shap_prompt manquant"
assert hasattr(module, 'generate_shap_explanation'), "✗ generate_shap_explanation manquant"
assert module.HF_MODEL    == "mistralai/Mistral-7B-Instruct-v0.2:featherless-ai", "✗ HF_MODEL"
assert module.HF_BASE_URL == "https://router.huggingface.co/v1",                  "✗ HF_BASE_URL"

print("✓ Import du module réussi — toutes les fonctions présentes")
print("\n  Utilisation en J3S2 :")
print("  from utils.llm_utils import generate_shap_explanation, HF_MODEL")


---
## Bloc Git — Commit & Push

### Commit Git

```bash
git add utils/llm_utils.py
git commit -m "feat(j2s4): LLM HF Mistral 7B — utils/llm_utils.py"
git push origin main
```

```bash
git log --oneline -3
```

**Résultat attendu :**
```
b3c4d5e feat(j2s4): LLM HF Mistral 7B — utils/llm_utils.py
a2b3c4d feat(j2s3): SHAP explicabilité — modèles sérialisés
9f8e7d6 feat(j2s2): modèles RF/LR/GB — comparaison 8 configs
```

> **Google Colab** : préfixer chaque commande avec `!`  
> `!git add utils/llm_utils.py`  
> `!git commit -m "feat(j2s4): LLM HF Mistral 7B — utils/llm_utils.py"`  
> `!git push origin main`

---
## Récapitulatif J2S4 — Ce que vous avez produit

| Étape | Action | Livrable |
|-------|--------|----------|
| **Cell 0** | Setup ROOT + pip + .env | Environnement prêt |
| **Bloc 1** | Test API HF Mistral 7B complet | Validation end-to-end |
| **Bloc 1b** | Diagnostic minimal (`max_tokens=5`) | Outil de débogage |
| **Bloc 2** | `build_shap_prompt()` + `generate_shap_explanation()` | Fonctions testées |
| **Bloc 3** | Export `utils/llm_utils.py` | Module réutilisable J3S2 |
| **Commit** | `feat(j2s4):` | Historique Git propre |

### Architecture LLM — Flux complet

```
Contributions SHAP (top 6)
    ↓
build_shap_prompt()          →  Prompt contextuel BCEAO (FR)
    ↓
generate_shap_explanation()  →  Appel router.huggingface.co/v1
    ↓                            Mistral-7B-Instruct-v0.2:featherless-ai
Explication texte français
    ↓
J3S2 — Streamlit             →  Affichage waterfall SHAP + texte LLM
```

### Token Hugging Face — Configuration

1. Aller sur `https://huggingface.co/settings/tokens`
2. Créer un token **Fine-grained** (lecture suffisante pour l'inférence)
3. VS Code : ajouter dans `.env` → `HF_TOKEN=hf_xxxxxxxxxxxx`
4. Colab : **Secrets** → clé `HF_TOKEN` → valeur `hf_xxxxxxxxxxxx`
5. **Ne jamais commiter le token** (`.env` protégé par `.gitignore`)

### Conformité BCEAO

> L'explication LLM est **complémentaire**, pas substitutive.  
> Le waterfall SHAP reste le livrable de conformité principal (Instruction n°026-2016).  
> Le texte LLM facilite la communication de la décision — sans remplacer  
> la traçabilité quantitative fournie par les valeurs SHAP individuelles.

### Prochaine session — J3S1 MLflow

> Les credentials DagsHub utilisés ici (même token HF) seront réutilisés  
> pour le **tracking MLflow distant** en J3S1 — aucune configuration supplémentaire.